In [34]:
import regex as re

In [23]:
class BasicTokenizer:
    def train(self, text, vocab_size, verbose=False):
        self.merges = dict()
        self.vocab = [bytes([i]) for i in range(256)]
        merge_num = vocab_size - 256
        byte_txt = list(text.encode("utf-8"))
        for i in range(merge_num):
            top_pair = count_pair(byte_txt)[0][0]
            self.vocab.append(self.vocab[top_pair[0]] + self.vocab[top_pair[1]])
            self.merges[top_pair] = i + 256
            byte_txt = self.merge(byte_txt, top_pair)
        
        self.rev_merges = {v: k for k, v in self.merges.items()}    

    def merge(self, text, pair):
        new_txt = []
        skip_next = False
        for i in range(len(text) - 1):
            if skip_next:
                skip_next = False
                continue
            if text[i:i+2] == list(pair):
                new_txt.append(self.merges[pair])
                skip_next = True
            else:
                new_txt.append(text[i])
        if not skip_next:
            new_txt.append(text[-1])
        return new_txt

    def encode(self, text):
        text = list(text.encode("utf-8"))
        for mer in self.merges:
            new_txt = []
            skip_next = False
            for pair in zip(text, text[1:]):
                if skip_next:
                    skip_next = False
                    continue
                if pair == mer:
                    new_txt.append(self.merges[pair])
                    skip_next = True
                else:
                    new_txt.append(pair[0])
            if not skip_next:
                new_txt.append(text[-1])
            text = new_txt.copy()

        return text

    def decode(self, ids):
        change = True
        while change:
            new_ids = []
            change = False
            for id in ids:
                if self.rev_merges.get(id):
                    new_ids += list(self.rev_merges[id])
                    change = True
                else:
                    new_ids.append(id)

            
            ids = new_ids.copy()
        text = bytes(ids).decode("utf-8")

        return text

    def visualizer_merges(self):
        print(bytes(self.merges[0][0]) + " + " + bytes(self.merges[0][1]))

In [30]:
tokenizer = BasicTokenizer()
tokenizer.train("hello uw ơ ????", 260)
tokenizer.vocab[256]
for pair, mer in tokenizer.merges.items():
    print(tokenizer.vocab[pair[0]], "+", tokenizer.vocab[pair[1]], "=", tokenizer.vocab[mer])

b'?' + b'?' = b'??'
b'h' + b'e' = b'he'
b'he' + b'l' = b'hel'
b'hel' + b'l' = b'hell'


In [4]:
def count_pair(text):
    """text in bytes"""
    count = dict()
    for a, b in zip(text, text[1:]):
        count[(a, b)] = count.get((a,b), 0) + 1

    return sorted(count.items(), key=lambda x: x[1], reverse=True)

In [32]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN

"'(?i:[sdmt]|ll|ve|re)|[^\\r\\n\\p{L}\\p{N}]?+\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]++[\\r\\n]*|\\s*[\\r\\n]|\\s+(?!\\S)|\\s+"

In [35]:
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

pattern = re.compile(GPT4_SPLIT_PATTERN)
text = "Archived from the original"
splitted = re.findall(pattern, text)
print(splitted)


['Archived', ' from', ' the', ' original']


In [ ]:
class RegexTokenizer:
    def train(self, text, vocab_size, verbose=False):
        self.merges = dict()
        self.vocab = [bytes([i]) for i in range(256)]
        merge_num = vocab_size - 256
        byte_txt = list(text.encode("utf-8"))
        for i in range(merge_num):
            top_pair = count_pair(byte_txt)[0][0]
            self.vocab.append(self.vocab[top_pair[0]] + self.vocab[top_pair[1]])
            self.merges[top_pair] = i + 256
            byte_txt = self.merge(byte_txt, top_pair)
        
        self.rev_merges = {v: k for k, v in self.merges.items()}